Realiza uma análise exploratória da camada Bronze para identificar inconsistências, valores ausentes, duplicidades e padrões de dados que devem ser considerados no processo de transformação e limpeza da camada Silver.

In [0]:
from pyspark.sql.functions import col, when, trim, to_timestamp, lit

df_vra = spark.table("voebem.bronze.vra")

# Valores que serão considerados como NULL
suspect_values = [
    "null", "NULL", "Null",
    "", "NaN", "nan",
    "None", "none",
    "NA", "na"
]

df_silver = df_vra

# Limpeza das colunas string
for c in df_silver.columns:

    if dict(df_silver.dtypes)[c] == "string":

        cleaned_col = trim(col(c))

        df_silver = df_silver.withColumn(
            c,
            when(
                cleaned_col.isNull() | cleaned_col.isin(suspect_values),
                None
            ).otherwise(cleaned_col)
        )

# Conversão das colunas de data/hora para TIMESTAMP
timestamp_cols = [
    "partida_prevista",
    "partida_real",
    "chegada_prevista",
    "chegada_real"
]

for c in timestamp_cols:

    if c in df_silver.columns:

        df_silver = df_silver.withColumn(
            c,
            to_timestamp(col(c))
        )

# Preenche valores ausentes de codigo_justificativa
if "codigo_justificativa" in df_silver.columns:

    df_silver = df_silver.withColumn(
        "codigo_justificativa",
        when(
            col("codigo_justificativa").isNull(),
            lit("N/A")
        ).otherwise(col("codigo_justificativa"))
    )

# Grava a camada Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("voebem.silver.vra")

In [0]:
from pyspark.sql.functions import col, unix_timestamp, to_date

df_silver = spark.table("voebem.silver.vra")

# Atraso de partida em minutos
df_silver = df_silver.withColumn(
    "atraso_partida_min",
    ((unix_timestamp("partida_real") -
      unix_timestamp("partida_prevista")) / 60).cast("int")
)

# Atraso de chegada em minutos
df_silver = df_silver.withColumn(
    "atraso_chegada_min",
    ((unix_timestamp("chegada_real") -
      unix_timestamp("chegada_prevista")) / 60).cast("int")
)

# Minutos recuperados durante o voo
df_silver = df_silver.withColumn(
    "minutos_recuperados",
    col("atraso_partida_min") -
    col("atraso_chegada_min")
)

# Cria colunas somente com a data,
# mantendo as colunas originais como TIMESTAMP
date_cols = [
    "partida_real",
    "partida_prevista",
    "chegada_real",
    "chegada_prevista"
]

for c in date_cols:

    if c in df_silver.columns:

        df_silver = df_silver.withColumn(
            c + "_data",
            to_date(col(c))
        )

# Grava novamente a camada Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("voebem.silver.vra")

Diferença 0 demonstrando que foi mantido a informação entre a camada bronze e silver

In [0]:
display(spark.sql("""
    SELECT
      (SELECT COUNT(*) FROM voebem.bronze.vra) AS bronze_vra,
      (SELECT COUNT(*) FROM voebem.silver.vra) AS silver_vra,
      (SELECT COUNT(*) FROM voebem.bronze.vra)
        - (SELECT COUNT(*) FROM voebem.silver.vra) AS diferenca
"""))

In [0]:
display(spark.sql("""
    SELECT icao_empresa_aerea, numero_voo, icao_aerodromo_origem, icao_aerodromo_destino,
           partida_prevista, partida_prevista_data, date_format(partida_prevista, 'HH:mm:ss') AS partida_prevista_hora,
           atraso_partida_min, atraso_chegada_min, minutos_recuperados, situacao_voo
    FROM voebem.silver.vra
    ORDER BY partida_prevista
    LIMIT 5
"""))

Unificação das tabelas de empresas estrangeiras e nacionais, especificando a origem

In [0]:
from pyspark.sql.functions import lit, current_timestamp

# Consolida as tabelas Bronze na camada Silver
spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.empresas AS

SELECT
    icao,
    estrangeira AS sigla_iata,
    razao AS razao_social,
    servico,
    cidade,
    uf,
    ativa AS situacao,
    'nacional' AS origem_cadastro,
    _arquivo_origem,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_nacionais

UNION ALL

SELECT
    icao,
    estrangeira AS sigla_iata,
    razao AS razao_social,
    servico,
    cidade,
    uf,
    ativa AS situacao,
    'estrangeira' AS origem_cadastro,
    _arquivo_origem,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_estrangeiras
""")

# Validação da quantidade de registros
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM voebem.bronze.empresas_nacionais) AS bronze_nacional,
    (SELECT COUNT(*) FROM voebem.bronze.empresas_estrangeiras) AS bronze_estrangeira,
    (SELECT COUNT(*) FROM voebem.bronze.empresas_nacionais)
      + (SELECT COUNT(*) FROM voebem.bronze.empresas_estrangeiras) AS bronze_total,
    (SELECT COUNT(*) FROM voebem.silver.empresas) AS silver_total
"""))

# Validação por origem e ICAO
display(spark.sql("""
SELECT
    origem_cadastro,
    COUNT(*) AS linhas,
    COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
FROM voebem.silver.empresas
GROUP BY origem_cadastro
ORDER BY origem_cadastro
"""))

Transformação das tabelas de aeródromos e códigos de operação, com padronização de colunas, conversão de tipos e inclusão do `_transformado_em`.

Ao final, é feita uma validação da quantidade de registros entre Bronze e Silver.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.aerodromos AS
SELECT
    codigo_oaci AS icao,
    ciad,
    nome,
    municipio,
    uf AS uf_nome,
    municipio_servido,
    uf_servido AS uf_servido_nome,
    latitude AS latitude_dms,
    longitude AS longitude_dms,
    try_cast(replace(altitude, ',', '.') AS DOUBLE) AS altitude_m,
    situacao,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.aerodromos
""")

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.codigos_operacao AS
SELECT
    dominio,
    codigo,
    descricao,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.codigos_operacao
""")

display(spark.sql("""
SELECT
    'aerodromos' AS tabela,
    (SELECT COUNT(*) FROM voebem.bronze.aerodromos) AS bronze,
    (SELECT COUNT(*) FROM voebem.silver.aerodromos) AS silver,
    (SELECT COUNT(*) FROM voebem.bronze.aerodromos)
      - (SELECT COUNT(*) FROM voebem.silver.aerodromos) AS diferenca

UNION ALL

SELECT
    'codigos_operacao',
    (SELECT COUNT(*) FROM voebem.bronze.codigos_operacao),
    (SELECT COUNT(*) FROM voebem.silver.codigos_operacao),
    (SELECT COUNT(*) FROM voebem.bronze.codigos_operacao)
      - (SELECT COUNT(*) FROM voebem.silver.codigos_operacao)
"""))

# Documentação de Metadados das Tabelas Silver

Aplica comentários descritivos em todas as colunas das tabelas `voebem.silver.vra`, `voebem.silver.empresas`, `voebem.silver.aerodromos` e `voebem.silver.codigos_operacao` usando `ALTER TABLE ... ALTER COLUMN ... COMMENT`. Os comentários explicam o significado de negócio de cada coluna e os campos de auditoria (`_arquivo_origem`, `_ingerido_em`, `_transformado_em`).

In [0]:
# Dicionário de comentários por tabela e coluna
comentarios = {
    "voebem.silver.vra": {
        "icao_empresa_aerea": "Código ICAO da empresa aérea operadora do voo",
        "numero_voo": "Número do voo",
        "codigo_autorizacao_di": "Código de autorização da etapa do voo",
        "codigo_tipo_linha": "Código que identifica o tipo de linha do voo",
        "icao_aerodromo_origem": "Código ICAO do aeródromo de origem",
        "icao_aerodromo_destino": "Código ICAO do aeródromo de destino",
        "partida_prevista": "Data e hora previstas para a partida do voo",
        "partida_real": "Data e hora reais da partida do voo",
        "chegada_prevista": "Data e hora previstas para a chegada do voo",
        "chegada_real": "Data e hora reais da chegada do voo",
        "situacao_voo": "Situação do voo",
        "codigo_justificativa": "Código de justificativa associado ao voo",
        "partida_prevista_data": "Data da partida prevista",
        "partida_real_data": "Data da partida real",
        "chegada_prevista_data": "Data da chegada prevista",
        "chegada_real_data": "Data da chegada real",
        "atraso_partida_min": "Atraso da partida em minutos",
        "atraso_chegada_min": "Atraso da chegada em minutos",
        "minutos_recuperados": "Minutos recuperados durante o voo",
        "_arquivo_origem": "Nome do arquivo CSV de origem na camada Bronze",
        "_ingerido_em": "Data e hora de ingestão do registro na camada Bronze",
    },

    "voebem.silver.empresas": {
        "icao": "Código ICAO da empresa aérea",
        "sigla_iata": "Sigla IATA da empresa aérea",
        "razao_social": "Razão social da empresa aérea",
        "servico": "Tipo de serviço prestado pela empresa",
        "cidade": "Cidade da empresa",
        "uf": "Unidade federativa da empresa",
        "situacao": "Situação cadastral da empresa",
        "origem_cadastro": "Origem do cadastro: nacional ou estrangeira",
        "_arquivo_origem": "Nome do arquivo de origem na camada Bronze",
        "_ingerido_em": "Data e hora de ingestão na camada Bronze",
        "_transformado_em": "Data e hora da transformação para a camada Silver",
    },

    "voebem.silver.aerodromos": {
        "icao": "Código ICAO do aeródromo",
        "ciad": "Código CIAD do aeródromo",
        "nome": "Nome do aeródromo",
        "municipio": "Município onde o aeródromo está localizado",
        "uf_nome": "Unidade federativa onde o aeródromo está localizado",
        "municipio_servido": "Município servido pelo aeródromo",
        "uf_servido_nome": "Unidade federativa do município servido",
        "latitude_dms": "Latitude em formato graus, minutos e segundos",
        "longitude_dms": "Longitude em formato graus, minutos e segundos",
        "altitude_m": "Altitude do aeródromo em metros",
        "situacao": "Situação do registro do aeródromo",
        "_ingerido_em": "Data e hora de ingestão na camada Bronze",
        "_transformado_em": "Data e hora da transformação para a camada Silver",
    },

    "voebem.silver.codigos_operacao": {
        "dominio": "Domínio ao qual o código de operação pertence",
        "codigo": "Código de operação",
        "descricao": "Descrição do código de operação",
        "_transformado_em": "Data e hora da transformação para a camada Silver",
    },
}

for tabela, colunas in comentarios.items():
    print(f"\nAplicando comentários em {tabela}...")

    for coluna, comentario in colunas.items():
        spark.sql(
            f"ALTER TABLE {tabela} "
            f"ALTER COLUMN {coluna} COMMENT '{comentario}'"
        )
        print(f"  ✓ {coluna}")

    print(f"  {len(colunas)} colunas documentadas.")

print("\nDocumentação de metadados concluída para todas as tabelas Silver.")

## Governança das tabelas Silver

Documentação e classificação das tabelas por meio de **comentários e tags**, incluindo camada, domínio, fonte e grão.

Também são realizadas auditorias para validar a **documentação das colunas** e as **tags aplicadas**.

In [0]:
%sql

-- =====================================================
-- GOVERNANÇA DAS TABELAS SILVER
-- =====================================================

-- Comentários descritivos nas tabelas
COMMENT ON TABLE voebem.silver.vra IS
  'Tabela de voos (VRA) tratada na camada Silver. Contém partidas, chegadas, atrasos e situacao de cada voo.';

COMMENT ON TABLE voebem.silver.empresas IS
  'Cadastro consolidado de empresas aereas nacionais e estrangeiras na camada Silver.';

COMMENT ON TABLE voebem.silver.aerodromos IS
  'Cadastro de aerodromos tratado na camada Silver, com coordenadas, altitude e situacao.';

COMMENT ON TABLE voebem.silver.codigos_operacao IS
  'Tabela de referencia de codigos de operacao (dominio, codigo e descricao) na camada Silver.';

-- Tags de governanca: camada, dominio, fonte, grao
ALTER TABLE voebem.silver.vra
SET TAGS (
    'camada' = 'silver',
    'dominio' = 'aviacao',
    'fonte' = 'ANAC-VRA',
    'grao' = 'etapa_de_voo'
);

ALTER TABLE voebem.silver.empresas
SET TAGS (
    'camada' = 'silver',
    'dominio' = 'aviacao',
    'fonte' = 'ANAC-empresas',
    'grao' = 'empresa'
);

ALTER TABLE voebem.silver.aerodromos
SET TAGS (
    'camada' = 'silver',
    'dominio' = 'aviacao',
    'fonte' = 'ANAC-aerodromos',
    'grao' = 'aerodromo'
);

ALTER TABLE voebem.silver.codigos_operacao
SET TAGS (
    'camada' = 'silver',
    'dominio' = 'aviacao',
    'fonte' = 'ANAC-codigos',
    'grao' = 'codigo'
);

-- =====================================================
-- AUDITORIA: PERCENTUAL DE COLUNAS DOCUMENTADAS
-- =====================================================
SELECT
    t.table_name,
    COUNT(c.column_name) AS total_colunas,
    SUM(CASE WHEN c.comment IS NOT NULL AND TRIM(c.comment) <> '' THEN 1 ELSE 0 END) AS colunas_documentadas,
    ROUND(
        100.0 * SUM(CASE WHEN c.comment IS NOT NULL AND TRIM(c.comment) <> '' THEN 1 ELSE 0 END)
        / COUNT(c.column_name),
        1
    ) AS pct_documentado
FROM system.information_schema.tables t
JOIN system.information_schema.columns c
    ON c.table_catalog = t.table_catalog
   AND c.table_schema  = t.table_schema
   AND c.table_name    = t.table_name
WHERE t.table_catalog = 'voebem'
  AND t.table_schema  = 'silver'
  AND t.table_name IN ('vra', 'empresas', 'aerodromos', 'codigos_operacao')
GROUP BY t.table_name
ORDER BY t.table_name;

-- =====================================================
-- AUDITORIA: TAGS APLICADAS NAS TABELAS SILVER
-- =====================================================
SELECT
    table_name,
    tag_name,
    tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'voebem'
  AND schema_name  = 'silver'
  AND table_name IN ('vra', 'empresas', 'aerodromos', 'codigos_operacao')
ORDER BY table_name, tag_name;

-- =====================================================
-- TABELAS EXISTENTES EM voebem.silver
-- =====================================================
SHOW TABLES IN voebem.silver;